# Embed TiniX chunks (bge-m3) on Colab GPU

Companion to `src/retrieval/embeddings.py` / `src/retrieval/vector_store.py` in the thesis repo. Moves the GPU-heavy embedding step off the local Mac (656,918 chunks was tying up the machine for hours) onto Colab's free GPU. Postgres/pgvector is only reachable from the local machine, so this notebook does **embedding only** - it writes a `chunks_embedded.jsonl` file, which you then download and load locally with `python3 -m scripts.upsert_precomputed_embeddings --input chunks_embedded.jsonl` (no model loading needed for that step, so it's fast even on CPU).

**Before running:** run `python3 -m scripts.split_chunks_for_upload` locally first - this splits `data/processed/chunks.jsonl` (635MB, too big to upload reliably over an unstable connection - repeated single-file uploads kept stalling partway) into ~45MB parts under `data/processed/chunks_parts/`. Upload **all** `chunks_part*.jsonl` files into a Google Drive folder at `MyDrive/thesis_embed/chunks_parts/` (each part is small enough to upload reliably on its own). This notebook reads every part file it finds there, so you don't need to reassemble them.

All progress is written straight to Drive too, so if Colab disconnects mid-run, just reopen this notebook and re-run - it resumes automatically from the last row it wrote, no re-embedding of already-done chunks.


## 1. Setup


In [3]:
!pip install -q sentence-transformers


In [4]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [5]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # must be set before torch touches CUDA
import torch, glob

# --- EDIT THESE if your Drive folder/file names differ ---
DRIVE_DIR = "/content/drive/MyDrive/thesis_embed"
INPUT_PARTS_DIR = f"{DRIVE_DIR}/chunks_parts"  # folder of chunks_part*.jsonl (see scripts/split_chunks_for_upload.py)
OUTPUT_PATH = f"{DRIVE_DIR}/chunks_embedded.jsonl"
BATCH_SIZE = 128  # raised for A100 (40GB VRAM, ~2.7x a T4's 14.56GB) - the T4 OOM'd
                  # at 64 with some large-table chunks. encode_safe() below auto-halves
                  # on OOM regardless of GPU, so this is safe to try aggressively; if you
                  # get reassigned a smaller GPU in a future session, drop this back down.
# ------------------------------------------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cpu":
    raise RuntimeError(
        "KHONG CO GPU. Vao menu Runtime > Change runtime type > chon GPU (T4), "
        "roi Runtime > Restart session, sau do chay lai TU CELL DAU TIEN.\n"
        "Chay embed tren CPU trong Colab thi khong co y nghia gi so voi chay local."
    )
print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")

input_files = sorted(glob.glob(f"{INPUT_PARTS_DIR}/chunks_part*.jsonl"))
if not input_files:
    raise FileNotFoundError(
        f"Khong tim thay file chunks_part*.jsonl nao trong {INPUT_PARTS_DIR}\n\n"
        f"Ban can, tren MAY LOCAL (khong phai Colab):\n"
        f"  1. Chay: python3 -m scripts.split_chunks_for_upload\n"
        f"     (tach data/processed/chunks.jsonl thanh cac file ~45MB trong data/processed/chunks_parts/)\n"
        f"  2. Mo drive.google.com (dung DUNG tai khoan Drive da mount o cell tren)\n"
        f"  3. Tao folder 'thesis_embed/chunks_parts' trong My Drive (neu chua co)\n"
        f"  4. Upload TOAN BO cac file chunks_part*.jsonl vao dung folder do (tung file mot cung duoc,\n"
        f"     moi file chi ~45MB nen upload on dinh hon 1 file 635MB duy nhat)\n"
        f"  5. Doi tat ca upload xong hoan toan (Drive UI het thanh tien trinh)\n"
        f"  6. Chay lai cell nay\n\n"
        f"Neu ban de file o ten/duong dan khac, sua INPUT_PARTS_DIR o tren cho khop."
    )
total_size_mb = sum(os.path.getsize(p) for p in input_files) / 1e6
print(f"Found {len(input_files)} part file(s), {total_size_mb:.1f} MB total - OK to proceed.")
for p in input_files:
    print(f"  {os.path.basename(p)}: {os.path.getsize(p)/1e6:.1f} MB")


device: cuda
GPU: NVIDIA A100-SXM4-80GB (85.1 GB)
Found 14 part file(s), 634.9 MB total - OK to proceed.
  chunks_part001.jsonl: 48.1 MB
  chunks_part002.jsonl: 48.1 MB
  chunks_part003.jsonl: 47.2 MB
  chunks_part004.jsonl: 48.5 MB
  chunks_part005.jsonl: 47.6 MB
  chunks_part006.jsonl: 47.4 MB
  chunks_part007.jsonl: 49.0 MB
  chunks_part008.jsonl: 49.6 MB
  chunks_part009.jsonl: 49.6 MB
  chunks_part010.jsonl: 49.6 MB
  chunks_part011.jsonl: 48.6 MB
  chunks_part012.jsonl: 47.3 MB
  chunks_part013.jsonl: 47.5 MB
  chunks_part014.jsonl: 6.8 MB


## 2. Load model (same model/normalization as the local pipeline - must match for the vectors to be comparable in pgvector)


In [6]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "BAAI/bge-m3"  # must match src/retrieval/embeddings.py
model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=device)
print("Model loaded.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Model loaded.


## 3. Resumable embed loop
Safe to interrupt (Colab disconnect, manual stop) and re-run this cell - it skips rows already written to `OUTPUT_PATH` on Drive.


In [7]:
import json, os

def key_of(r):
    return (r["ticker"], r["source_file"], r["page"], r["chunk_index"])

done_keys = set()
if os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                done_keys.add(key_of(json.loads(line)))
            except Exception:
                pass
print(f"Already embedded (resume point): {len(done_keys)}")

records = []
for part_path in input_files:  # from the config cell above - all chunks_part*.jsonl found on Drive
    with open(part_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
print(f"Total records across {len(input_files)} part file(s): {len(records)}")

todo = [r for r in records if key_of(r) not in done_keys]
print(f"Remaining to embed this session: {len(todo)}")


Already embedded (resume point): 7296
Total records across 14 part file(s): 656918
Remaining to embed this session: 649622


In [8]:
import torch

def encode_safe(texts, batch_size):
    """Encode `texts` with automatic recovery from CUDA OOM: on failure, clears the
    cache and retries as two smaller sub-batches (halving batch_size each time), down
    to batch_size=1 if needed. Isolates whichever specific chunk(s) are too large
    (long financial tables) into their own tiny sub-batch instead of failing the
    whole run - preserves the full chunk text (no truncation), just processes it
    with less parallelism."""
    try:
        return list(model.encode(texts, batch_size=max(1, batch_size), normalize_embeddings=True))
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(texts) == 1:
            raise  # a single chunk alone doesn't fit - genuinely too long, will show in output
        mid = len(texts) // 2
        print(f"    [OOM] splitting batch of {len(texts)} into {mid}+{len(texts)-mid} and retrying...")
        left = encode_safe(texts[:mid], max(1, batch_size // 2))
        right = encode_safe(texts[mid:], max(1, batch_size // 2))
        return left + right


In [9]:
import time

out_f = open(OUTPUT_PATH, "a", encoding="utf-8")
n_done = 0
t0 = time.time()
try:
    for i in range(0, len(todo), BATCH_SIZE):
        batch = todo[i:i + BATCH_SIZE]
        texts = [r["chunk_text"] for r in batch]
        vecs = encode_safe(texts, BATCH_SIZE)
        for r, v in zip(batch, vecs):
            out = dict(r)
            out["embedding"] = [round(float(x), 6) for x in v]  # 6dp: matches local _vector_literal, keeps file size sane
            out_f.write(json.dumps(out, ensure_ascii=False) + "\n")
        out_f.flush()
        os.fsync(out_f.fileno())  # force the write to actually land on Drive, not just OS buffer
        n_done += len(batch)
        if (i // BATCH_SIZE) % 20 == 0:
            torch.cuda.empty_cache()  # periodic hygiene on a long run
            elapsed = time.time() - t0
            rate = n_done / elapsed if elapsed > 0 else 0
            remaining = len(todo) - n_done
            eta_sec = remaining / rate if rate > 0 else float("inf")
            eta_str = f"{eta_sec/60:.0f} phut" if eta_sec < 3600 else f"{eta_sec/3600:.1f} gio"
            print(f"  embedded {n_done}/{len(todo)} this session "
                  f"(total on disk: {len(done_keys) + n_done}/{len(records)}) | "
                  f"{rate:.1f} chunk/s | ETA con lai: {eta_str}")
finally:
    out_f.close()
elapsed_total = time.time() - t0
print(f"Session done in {elapsed_total/60:.1f} phut. {n_done} chunk(s) embedded and saved to Drive this run.")
print(f"Total progress: {len(done_keys) + n_done}/{len(records)}")
if len(done_keys) + n_done < len(records):
    print("Not finished yet - re-run the two cells above (or just this one) to continue.")
else:
    print("ALL DONE. Run the download cell below, then the local upsert script.")


  embedded 128/649622 this session (total on disk: 7424/656918) | 6.7 chunk/s | ETA con lai: 26.8 gio
  embedded 2688/649622 this session (total on disk: 9984/656918) | 35.7 chunk/s | ETA con lai: 5.0 gio
  embedded 5248/649622 this session (total on disk: 12544/656918) | 36.2 chunk/s | ETA con lai: 4.9 gio
  embedded 7808/649622 this session (total on disk: 15104/656918) | 36.4 chunk/s | ETA con lai: 4.9 gio
  embedded 10368/649622 this session (total on disk: 17664/656918) | 38.7 chunk/s | ETA con lai: 4.6 gio
  embedded 12928/649622 this session (total on disk: 20224/656918) | 35.8 chunk/s | ETA con lai: 4.9 gio
  embedded 15488/649622 this session (total on disk: 22784/656918) | 37.6 chunk/s | ETA con lai: 4.7 gio
  embedded 18048/649622 this session (total on disk: 25344/656918) | 38.6 chunk/s | ETA con lai: 4.5 gio
  embedded 20608/649622 this session (total on disk: 27904/656918) | 39.3 chunk/s | ETA con lai: 4.4 gio
  embedded 23168/649622 this session (total on disk: 30464/656

In [10]:
# Compress + download straight from this notebook (no manual Drive UI needed).
# Uncompressed output can be several GB (1024-dim float embeddings inflate the
# text ~10x) - gzip cuts that substantially before the download.
import gzip, shutil
from google.colab import files

GZ_PATH = OUTPUT_PATH + ".gz"
print("Compressing... (can take a minute for a large file)")
with open(OUTPUT_PATH, "rb") as f_in, gzip.open(GZ_PATH, "wb") as f_out:
    shutil.copyfileobj(f_in, f_out)

import os
print(f"Uncompressed: {os.path.getsize(OUTPUT_PATH)/1e9:.2f} GB")
print(f"Compressed:   {os.path.getsize(GZ_PATH)/1e9:.2f} GB")

files.download(GZ_PATH)
print("Download triggered - check your browser's download bar.")
print("If the browser download stalls on a big file, the .gz is also sitting in")
print(f"Drive at {GZ_PATH} - open drive.google.com and download it from there instead.")


Compressing... (can take a minute for a large file)
Uncompressed: 7.64 GB
Compressed:   2.28 GB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download triggered - check your browser's download bar.
If the browser download stalls on a big file, the .gz is also sitting in
Drive at /content/drive/MyDrive/thesis_embed/chunks_embedded.jsonl.gz - open drive.google.com and download it from there instead.


## 4. Get the file back onto your Mac

Two options once `Total progress` above equals the full record count:

**Option A - straight from this notebook (cell above):** compresses the output and triggers a normal browser download via `google.colab.files.download()`. Simplest if it doesn't stall (Colab's download bridge can be flaky for multi-GB files on some networks).

**Option B - via Drive directly (fallback, more reliable for big files):** open [drive.google.com](https://drive.google.com), go to `MyDrive/thesis_embed/`, right-click `chunks_embedded.jsonl` (or the `.jsonl.gz` if you ran the compress cell) → Download. If you have the Google Drive desktop app installed and synced, the file just appears under `~/Google Drive/My Drive/thesis_embed/` in Finder automatically - no manual download step at all.

Then on your Mac, in the thesis repo (the local script accepts both `.jsonl` and `.jsonl.gz` directly - no need to unzip first):
```
python3 -m scripts.upsert_precomputed_embeddings --input /path/to/chunks_embedded.jsonl.gz
```
This only writes to Postgres - no model loading, fast even on CPU.
